In [11]:
import kagglehub
import pandas as pd
import numpy as np
# Download latest version
# path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")

# print("Path to dataset files:", path)

In [12]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [13]:
import sqlite3

conn = sqlite3.connect("f1_data.db")

csv_files = ["circuits.csv","constructor_results.csv","constructor_standings.csv",
             "drivers.csv", "races.csv", "results.csv", "constructors.csv",
             "driver_standings.csv",'drivers.csv','lap_times.csv',
             "pit_stops.csv","qualifying.csv","races.csv",'results.csv','seasons.csv',
             "sprint_results.csv",'status.csv']
csv_files_path = ["data" +"\\"+ c for c in csv_files]
for file in csv_files_path:
    df = pd.read_csv(file)
    # Clean column names (remove spaces/special chars for SQL safety)
    df.columns = [c.replace(' ', '_').lower() for c in df.columns]
    table_name = file.replace(".csv", "")
    df.to_sql(table_name, conn, if_exists="replace", index=False)

# Version 1: Multi-table SQLite LangGraph pipeline

Architecture:

User input -> Generate SQL query -> Fetch data from SQLite -> Summarize result -> Return answer

In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2
)

In [15]:
class State(TypedDict):
    user_query:str
    sql_code:str
    sql_code_answer:str
    summary:str

Define the graph state and node functions for the V1 pipeline.

In [16]:
# V1 uses plain LangGraph node functions, not LangChain tool wrappers.

The nodes below generate SQL, execute it against the local SQLite database, and summarize the result.

In [17]:
conn = sqlite3.connect('f1_data.db')
cursor = conn.cursor()


table_names = [c.replace(".csv","") for c in csv_files]
schema_info_all = []
for table in table_names:
    cursor.execute(f"PRAGMA table_info({table})")

    columns = cursor.fetchall()
    schema_info_all.append([[col[1],col[2]] for col in columns])

table_schema_info={}
for i in range(len(table_names)):
        table_schema_info[table_names[i]] = schema_info_all[i]


def query_generator(state:State):
    """Function for transforming user input into SQL query"""
    prompt = f"""Generate SQL code for this query - {state['user_query']}, using the provided context in a dictionary: 
    where the keys are table_names and the values are the schema in a list form.
    {table_schema_info}. Respond only with the SQL query."""
    output = llm.invoke(prompt)
    return {'sql_code':output.content}


def run_sqlite_query(state:State):
    """
    Executes a SQL query against the F1 dataset and returns the results.
    Use this to join tables or aggregate F1 statistics.
    """
    conn = sqlite3.connect("f1_data.db")
    try:
        cursor = conn.cursor()
        cursor.execute(state['sql_code'])
        result = cursor.fetchall()
        return {'sql_code_answer':result}
    except Exception as e:
        return {'sql_code_answer': f"Error: {str(e)}"}
    finally:
        conn.close()
        


def summarizer(state:State):
    """ Provides a proper summary for the SQL query and its answer for the user."""
    prompt = f"""SYSTEM INSTRUCTION:
            You are an F1 Data Analyst. Your task is to take a user's question and the 
            corresponding SQL data results to provide a polished, expert response.

            INPUT DATA:
            1. User Question: {state['user_query']}
            2. Executed SQL: {state['sql_code']}
            3. SQL Result: {state['sql_code_answer']}

            GUIDELINES:
            - Directness: Answer the user's question in the first sentence.
            - Accuracy: Use ONLY the data provided in the 'SQL Result'. Do not hallucinate external stats.
            - Professionalism: If the result is a list of names or numbers, format them clearly (e.g., use bullet points for multiple drivers).
            - Context: If the SQL result is empty, politely explain that no records were found for that specific query in the F1 database.
            - Detail: Mention the specific driver names and values exactly as they appear in the result.

            RESPONSE FORMAT:
            "Based on the race data, [Your natural language answer here]."
            """
    output = llm.invoke(prompt)
    return {'summary':output.content}


In [18]:
builder = StateGraph(State)
builder.add_node("query generator",query_generator)
builder.add_node("sqlite3 generator",run_sqlite_query)
builder.add_node("summarization",summarizer)
builder.set_entry_point("query generator")
builder.add_edge("query generator", "sqlite3 generator")
builder.add_edge("sqlite3 generator","summarization")
builder.add_edge("summarization",END)
graph = builder.compile()

result = graph.invoke({"user_query":"tell me who had the most DNFs in 2023?"})

In [19]:
result

{'user_query': 'tell me who had the most DNFs in 2023?',
 'sql_code': "SELECT d.driverid,\n       d.forename,\n       d.surname,\n       COUNT(*) AS dnf_count\nFROM results r\nJOIN races ra ON r.raceid = ra.raceid\nJOIN status s ON r.statusid = s.statusid\nJOIN drivers d ON r.driverid = d.driverid\nWHERE ra.year = 2023\n  AND s.status NOT LIKE 'Finished%'\nGROUP BY d.driverid, d.forename, d.surname\nORDER BY dnf_count DESC\nLIMIT 1;",
 'sql_code_answer': [(825, 'Kevin', 'Magnussen', 15)],
 'summary': 'Based on the race data, Kevin\u202fMagnussen had the most DNFs in 2023 with 15.'}

# Version 2:

1. Adding a routing decision

    - Setting up multiple subagents.
        - One for factual lookup
        - One for comparative/deeper analysis (over a season or multiple seasons)
        - Hybrid (lookup/analysis + structured summary)

2. Improved the system prompt for the LLM
3. Added PydanticAI (structured output) to ensure LLM always returns answers in one format. 

In [27]:
class State(TypedDict):
    user_query:str
    intent:str
    sql_code:str
    summary:str

In [21]:
from pydantic import BaseModel, Field
from typing import Literal

In [ ]:
from ast import List


class intent_classification(BaseModel):
    user_query: str = Field(description="The user query")
    intent: Literal["Factual Lookup","Deep Analysis","Hybrid"]

structuredllm = llm.with_structured_output(intent_classification)

In [28]:
def query_generator_v2(state:State):
    prompt = f"""You are an F1 expert. You have access to a SQLite database containing information
    about all the races from 1950 to 2024. Your task is to classify intent and generate SQL queries using this 
    schema that is tailored to answer the user's question correctly. Below is the schema and the user question:
    Context = {table_schema_info}
    User Question = {state['user_query']}
    Intent: "Factual Lookup" or "Deep Analysis" or "Hybrid"
    ########################################
    
    RULES
    1. Always generate SQL code for the "User Question".
    2. Ensure the SQL code is accurate and verified.
    3. Assume columns with same name across tables mean the same thing.
    4. For each "User Question", classify it as one of three types:
        - Factual Lookup: Use this if the question has a single, verifiable answer found in the data (e.g., "What was the date of...", "Who won...").
        - Deep Analysis: Use this if the user wants an interpretation, a trend, or a statistical breakdown (e.g., "Why did...", "Analyze the trend...", "Was X better than Y?").
        - Hybrid: Use this if the user asks for a specific fact AND an analysis of that fact in one go.
    """
    response = structuredllm.invoke(prompt)
    return {'sql_code':response.sql_code,
            'sql_code_type':response.intent
            }

In [29]:
def routing_decision(state:State):
    if state['intent']=='Factual Lookup':
        return 1
    elif state['intent']=='Deep Analysis':
        return 2
    else:
        return 0

In [ ]:
def analyst(state:State):
    prompt = f"""
        
    """

In [ ]:
builder = StateGraph(State)
builder.add_node("query generator",query_generator)
builder.add_node("sqlite3 generator",run_sqlite_query)
builder.add_node("summarization",summarizer)
builder.set_entry_point("query generator")
builder.add_edge("query generator", "sqlite3 generator")
builder.add_edge("sqlite3 generator","summarization")
builder.add_edge("summarization",END)
graph = builder.compile()

result = graph.invoke({"user_query":"tell me who had the most DNFs in 2023?"})